<a href="https://colab.research.google.com/github/MakhloufiMahdi/customer-churn-intelligence-platform/blob/main/02_data_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# URL du dataset IBM Telco Customer Churn
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# Charger les données brutes
df_raw = pd.read_csv(url)

print("Dataset chargé avec succès !")
print("Dimensions :", df_raw.shape)

display(df_raw.head())

Dataset chargé avec succès !
Dimensions : (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df_clean = df_raw.copy()

print("Copie créée :", df_clean.shape)

Copie créée : (7043, 21)


In [4]:
df_clean["TotalCharges"] = (
    df_clean["TotalCharges"]
    .astype(str)
    .str.strip()
    .replace("", np.nan)
)

df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce"
)

print(
    "Valeurs manquantes dans TotalCharges :",
    df_clean["TotalCharges"].isna().sum()
)

Valeurs manquantes dans TotalCharges : 11


In [5]:
condition = (
    df_clean["TotalCharges"].isna()
    & (df_clean["tenure"] == 0)
)

print("Clients concernés :", condition.sum())

df_clean.loc[condition, "TotalCharges"] = 0

print(
    "Valeurs manquantes restantes :",
    df_clean["TotalCharges"].isna().sum()
)

Clients concernés : 11
Valeurs manquantes restantes : 0


In [6]:
quality_report = pd.DataFrame({
    "dtype": df_clean.dtypes.astype(str),
    "missing_values": df_clean.isna().sum(),
    "unique_values": df_clean.nunique()
})

display(quality_report)

print("Dimensions :", df_clean.shape)
print("Lignes dupliquées :", df_clean.duplicated().sum())
print(
    "Identifiants clients dupliqués :",
    df_clean["customerID"].duplicated().sum()
)

print("\nStatistiques de TotalCharges :")
display(df_clean["TotalCharges"].describe())

,dtype,missing_values,unique_values
customerID,object,0,7043
gender,object,0,2
SeniorCitizen,int64,0,2
Partner,object,0,2
Dependents,object,0,2
tenure,int64,0,73
PhoneService,object,0,2
MultipleLines,object,0,3
InternetService,object,0,3
OnlineSecurity,object,0,3


Dimensions : (7043, 21)
Lignes dupliquées : 0
Identifiants clients dupliqués : 0

Statistiques de TotalCharges :


,TotalCharges
count,7043.000000
mean,2279.734304
std,2266.794470
min,0.000000
25%,398.550000
50%,1394.550000
75%,3786.600000
max,8684.800000


In [7]:
assert df_clean.shape == (7043, 21)
assert df_clean["customerID"].nunique() == 7043
assert df_clean.duplicated().sum() == 0
assert df_clean["TotalCharges"].isna().sum() == 0
assert pd.api.types.is_numeric_dtype(df_clean["TotalCharges"])
assert (df_clean["TotalCharges"] >= 0).all()
assert (df_clean["tenure"] >= 0).all()

print("Tous les contrôles qualité sont validés.")

Tous les contrôles qualité sont validés.


In [8]:
output_path = "/content/telco_churn_clean.csv"

df_clean.to_csv(output_path, index=False)

print("Dataset sauvegardé :", output_path)

Dataset sauvegardé : /content/telco_churn_clean.csv


In [9]:
import os

print("Fichier existe :", os.path.exists(output_path))
print("Taille :", round(os.path.getsize(output_path) / 1024, 2), "KB")

Fichier existe : True
Taille : 949.13 KB
